# 18 — P2: Hybrid Verifier (Lipschitz → IBP → PGD → MILP)

**Plan 2 — Phase 6.** Compositional verifier that runs cheap-and-loose
methods first and only escalates to MILP for samples that survive the
cascade.

```
hybrid_verify(model, x0, ε, y_true, L_global) →
    1. Lipschitz pre-filter
       margin = clean_margin(model, x0, y_true)
       if margin > L_global · ε · √(D)  → return verified
    2. IBP propagation (ms)
       if IBP-certified                  → return verified
    3. PGD attack (seconds)
       if attack succeeds                 → return falsified (with adversarial)
    4. MILP (minutes)
       return verify_vit_milp(...)
```

Per Plan 2 §6: each stage runs only if the previous fails. Soundness is
guaranteed because Lipschitz / IBP / MILP are all sound, and PGD only ever
returns *falsified* (never *verified*).

## Inputs / outputs
- 2 trained ViTs (standard + lipmargin) loaded from
  `/content/drive/My Drive/thesis-formal-verification/runs/vit_tiny_*/model.pt`
- Lipschitz constants from notebook 11 (`results/vit_p2/lipschitz_bounds.json`)
- 50 MNIST eval samples (deterministic seed=1234)
- ε ∈ {0.01, 0.03, 0.1}
- Output: `results/vit_p2/hybrid_results.csv` with one row per
  `(model, ε, sample)` recording status, resolving method, time per stage.

## Notes on the trained models
The standard ViT solves quickly via PGD (verifies almost nothing — see
notebook 13: PGD-robust=0/50 at ε=0.1). The lipmargin model has very low
clean accuracy in the current checkpoint (notebook 13: clean=3/50), so the
hybrid will skip most samples as misclassified.

MILP timeout per sample defaults to **120 s** (vs Plan 2's suggested 300 s)
to bound notebook runtime. Adjust `MILP_TIMEOUT` for production runs.


In [1]:
!pip install -q numpy pandas torch torchvision tqdm gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 111.6 MB/s eta 0:00:0000:010:01


In [2]:
from google.colab import drive
import os

drive.mount('/content/drive')

os.environ['GRB_LICENSE_FILE'] = '/content/drive/My Drive/thesis-formal-verification/gurobi.lic'
os.environ['PATH'] = '/content/drive/My Drive/thesis-formal-verification:' + os.environ.get('PATH', '')

print('Google Drive mounted and Gurobi path added to os.environ')

Mounted at /content/drive
Google Drive mounted and Gurobi path added to os.environ


In [3]:
import gurobipy as gp
from gurobipy import GRB
HAS_GUROBI = True
# Confirm the paid license is picked up — should NOT say "Restricted license":
gp.Model('lic_check').dispose()

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2819113
Academic license 2819113 - for non-commercial use only - registered to m____@ma.iitr.ac.in


In [4]:
from __future__ import annotations
import math, json, time, csv, warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, List, Tuple, Optional, Dict, Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision, torchvision.transforms as transforms

try:
    import gurobipy as gp
    from gurobipy import GRB
    HAS_GUROBI = True
except Exception as e:
    HAS_GUROBI = False
    print(f'Gurobi not available ({e}); MILP stage will be skipped.')

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(1234); np.random.seed(1234)
print(f'Device: {device}  Gurobi={HAS_GUROBI}')

Device: cuda  Gurobi=True


In [5]:
# ── ViT-Tiny architecture (matches notebooks 09-16) ─────────────────────
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.dim, self.eps = dim, eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.weight * (x / rms)

class PatchEmbed(nn.Module):
    def __init__(self, img_size=28, patch_size=4, in_channels=1, embed_dim=64):
        super().__init__(); self.img_size, self.patch_size = img_size, patch_size
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)
    def forward(self, x): return self.proj(x).flatten(2).transpose(1, 2)

class MHSA(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.embed_dim, self.num_heads = embed_dim, num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=True)
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=True)
    def forward(self, x):
        B, N, C = x.shape; H, D = self.num_heads, self.head_dim
        q = self.W_q(x).view(B, N, H, D).transpose(1, 2)
        k = self.W_k(x).view(B, N, H, D).transpose(1, 2)
        v = self.W_v(x).view(B, N, H, D).transpose(1, 2)
        s = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        a = F.softmax(s, dim=-1)
        o = torch.matmul(a, v).transpose(1, 2).contiguous().view(B, N, C)
        return self.W_o(o)

class MLPBlock(nn.Module):
    def __init__(self, embed_dim, mlp_ratio=2):
        super().__init__(); h = embed_dim * mlp_ratio
        self.fc1 = nn.Linear(embed_dim, h); self.fc2 = nn.Linear(h, embed_dim)
        self.act = nn.ReLU()
    def forward(self, x): return self.fc2(self.act(self.fc1(x)))

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=2, eps_rms=1e-6):
        super().__init__()
        self.norm1 = RMSNorm(embed_dim, eps_rms)
        self.attn  = MHSA(embed_dim, num_heads)
        self.norm2 = RMSNorm(embed_dim, eps_rms)
        self.mlp   = MLPBlock(embed_dim, mlp_ratio)
    def forward(self, x):
        x = x + self.attn(self.norm1(x)); x = x + self.mlp(self.norm2(x)); return x

class ViTTiny(nn.Module):
    def __init__(self, img_size=28, patch_size=4, in_channels=1, num_classes=10,
                 embed_dim=64, num_heads=2, num_layers=2, mlp_ratio=2, eps_rms=1e-6):
        super().__init__()
        self.cfg = dict(img_size=img_size, patch_size=patch_size,
                        in_channels=in_channels, num_classes=num_classes,
                        embed_dim=embed_dim, num_heads=num_heads,
                        num_layers=num_layers, mlp_ratio=mlp_ratio, eps_rms=eps_rms)
        self.patch_embed = PatchEmbed(img_size, patch_size, in_channels, embed_dim)
        self.pos_embed   = nn.Parameter(torch.zeros(1, self.patch_embed.n_patches, embed_dim))
        self.blocks = nn.ModuleList([TransformerBlock(embed_dim, num_heads, mlp_ratio, eps_rms)
                                     for _ in range(num_layers)])
        self.norm = RMSNorm(embed_dim, eps_rms)
        self.head = nn.Linear(embed_dim, num_classes)
    def forward(self, x):
        x = self.patch_embed(x) + self.pos_embed
        for blk in self.blocks: x = blk(x)
        return self.head(self.norm(x).mean(dim=1))

def load_vit(ckpt_path: Path, device='cpu') -> ViTTiny:
    payload = torch.load(ckpt_path, map_location=device, weights_only=False)
    m = ViTTiny(**payload['cfg']).to(device)
    sd = payload.get('state_dict_materialized', payload['state_dict'])
    sd = {k: v for k, v in sd.items() if 'parametrizations' not in k}
    m.load_state_dict(sd, strict=False); m.eval()
    return m

In [6]:
# ── PWL bracket + Gurobi big-M encoder + McCormick (notebooks 14-16) ────
@dataclass
class PWLBracket:
    name: str; breakpoints: List[float]
    slope_lo: List[float]; int_lo: List[float]
    slope_up: List[float]; int_up: List[float]
    @property
    def n_pieces(self): return len(self.breakpoints) - 1
    @property
    def domain(self):   return (self.breakpoints[0], self.breakpoints[-1])

def _line(p1, p2):
    s = (p2[1]-p1[1])/(p2[0]-p1[0]); return s, p1[1]-s*p1[0]
def _tan(f, df, x0):
    s = df(x0); return s, f(x0)-s*x0

def build_pwl_convex(f, df, a, b, n, name):
    bps = list(np.linspace(a, b, n+1))
    sl, il, su, iu = [], [], [], []
    for k in range(n):
        x_l, x_r = bps[k], bps[k+1]
        s_u, i_u = _line((x_l,f(x_l)),(x_r,f(x_r))); su.append(s_u); iu.append(i_u)
        s_l, i_l = _tan(f, df, 0.5*(x_l+x_r));       sl.append(s_l); il.append(i_l)
    return PWLBracket(name, bps, sl, il, su, iu)

def pwl_square(a, b, n=8): return build_pwl_convex(lambda x: x*x, lambda x: 2*x, a, b, n, 'sq')
def pwl_exp(a, b, n=8):     return build_pwl_convex(math.exp, math.exp, a, b, n, 'exp')
def pwl_inv_pos(a, b, n=8):
    assert a > 0
    return build_pwl_convex(lambda x: 1.0/x, lambda x: -1.0/(x*x), a, b, n, 'inv')
def pwl_inv_sqrt_pos(a, b, n=8):
    assert a > 0
    return build_pwl_convex(lambda t: 1.0/math.sqrt(t),
                            lambda t: -0.5*t**(-1.5), a, b, n, 'isq')

def add_pwl_bracket(model, x_var, y_var, br: PWLBracket, big_M=None, prefix=''):
    n, bps = br.n_pieces, br.breakpoints
    a, b = bps[0], bps[-1]
    if big_M is None:
        scope = max(
            max(abs(br.slope_lo[k])*(b-a)+abs(br.int_lo[k]) for k in range(n)),
            max(abs(br.slope_up[k])*(b-a)+abs(br.int_up[k]) for k in range(n)),
        )
        big_M = max(2.0*scope, 1.0)
    delta = [model.addVar(vtype=GRB.BINARY, name=f'{prefix}d_{k}') for k in range(n)]
    model.addConstr(gp.quicksum(delta) == 1)
    for k in range(n):
        x_l, x_r = bps[k], bps[k+1]
        model.addConstr(x_var >= x_l - big_M*(1-delta[k]))
        model.addConstr(x_var <= x_r + big_M*(1-delta[k]))
        model.addConstr(y_var >= br.slope_lo[k]*x_var + br.int_lo[k] - big_M*(1-delta[k]))
        model.addConstr(y_var <= br.slope_up[k]*x_var + br.int_up[k] + big_M*(1-delta[k]))

def add_mccormick(model, a_var, b_var, a_l, a_u, b_l, b_u, prefix='', y_lb=None, y_ub=None):
    z_lo = min(a_l*b_l, a_l*b_u, a_u*b_l, a_u*b_u)
    z_up = max(a_l*b_l, a_l*b_u, a_u*b_l, a_u*b_u)
    if y_lb is not None: z_lo = max(z_lo, y_lb)
    if y_ub is not None: z_up = min(z_up, y_ub)
    z = model.addVar(lb=z_lo, ub=z_up, name=f'{prefix}z')
    model.addConstr(z >= a_l*b_var + b_l*a_var - a_l*b_l)
    model.addConstr(z >= a_u*b_var + b_u*a_var - a_u*b_u)
    model.addConstr(z <= a_u*b_var + b_l*a_var - a_u*b_l)
    model.addConstr(z <= a_l*b_var + b_u*a_var - a_l*b_u)
    return z

def add_relu_bigM(model, x_var, x_lo, x_up, prefix=''):
    if x_lo >= 0:
        y = model.addVar(lb=x_lo, ub=x_up, name=f'{prefix}y')
        model.addConstr(y == x_var); return y
    if x_up <= 0:
        y = model.addVar(lb=0.0, ub=0.0, name=f'{prefix}y')
        model.addConstr(y == 0.0); return y
    delta = model.addVar(vtype=GRB.BINARY, name=f'{prefix}a')
    y = model.addVar(lb=0.0, ub=x_up, name=f'{prefix}y')
    model.addConstr(y >= x_var)
    model.addConstr(y <= x_up * delta)
    model.addConstr(y <= x_var - x_lo * (1 - delta))
    return y

In [7]:
# ── IBP propagation (numpy) for full ViT ────────────────────────────────
def ibp_lin(x_l, x_u, W, b):
    Wp = np.maximum(W, 0); Wn = np.minimum(W, 0)
    y_l = x_l @ Wp.T + x_u @ Wn.T
    y_u = x_u @ Wp.T + x_l @ Wn.T
    if b is not None: y_l = y_l + b; y_u = y_u + b
    return y_l, y_u

def ibp_relu(x_l, x_u): return np.maximum(x_l, 0), np.maximum(x_u, 0)

def ibp_rmsnorm(x_l, x_u, gamma, eps):
    sq_lo = np.where((x_l <= 0) & (x_u >= 0), 0.0, np.minimum(x_l**2, x_u**2))
    sq_up = np.maximum(x_l**2, x_u**2)
    msq_l = sq_lo.mean(axis=-1, keepdims=True)
    msq_u = sq_up.mean(axis=-1, keepdims=True)
    inv_l = 1.0 / np.sqrt(msq_u + eps)
    inv_u = 1.0 / np.sqrt(np.maximum(msq_l, 0) + eps)
    inv_l_b = np.broadcast_to(inv_l, x_l.shape)
    inv_u_b = np.broadcast_to(inv_u, x_u.shape)
    c1 = x_l*inv_l_b; c2 = x_l*inv_u_b; c3 = x_u*inv_l_b; c4 = x_u*inv_u_b
    z_l = np.minimum(np.minimum(c1,c2), np.minimum(c3,c4))
    z_u = np.maximum(np.maximum(c1,c2), np.maximum(c3,c4))
    gp_, gn_ = np.maximum(gamma, 0), np.minimum(gamma, 0)
    y_l = z_l*gp_ + z_u*gn_; y_u = z_u*gp_ + z_l*gn_
    return y_l, y_u

def ibp_attention_bounds(x_lo, x_up, attn):
    N, E = x_lo.shape; H, D = attn.num_heads, attn.head_dim
    scale = attn.scale
    Wq = attn.W_q.weight.detach().cpu().numpy()
    Wk = attn.W_k.weight.detach().cpu().numpy()
    Wv = attn.W_v.weight.detach().cpu().numpy()
    bv = attn.W_v.bias.detach().cpu().numpy() if attn.W_v.bias is not None else None
    Wo = attn.W_o.weight.detach().cpu().numpy()
    bo = attn.W_o.bias.detach().cpu().numpy() if attn.W_o.bias is not None else None
    Q_l, Q_u = ibp_lin(x_lo, x_up, Wq, None)
    K_l, K_u = ibp_lin(x_lo, x_up, Wk, None)
    V_l, V_u = ibp_lin(x_lo, x_up, Wv, bv)
    def to_h(a): return a.reshape(N, H, D).transpose(1, 0, 2)
    Q_l, Q_u = to_h(Q_l), to_h(Q_u)
    K_l, K_u = to_h(K_l), to_h(K_u)
    V_l, V_u = to_h(V_l), to_h(V_u)
    Q_e_l = Q_l[:, :, None, :]; Q_e_u = Q_u[:, :, None, :]
    K_e_l = K_l[:, None, :, :]; K_e_u = K_u[:, None, :, :]
    c1 = Q_e_l*K_e_l; c2 = Q_e_l*K_e_u; c3 = Q_e_u*K_e_l; c4 = Q_e_u*K_e_u
    P_l = np.minimum(np.minimum(c1,c2), np.minimum(c3,c4))
    P_u = np.maximum(np.maximum(c1,c2), np.maximum(c3,c4))
    S_l = scale * P_l.sum(-1); S_u = scale * P_u.sum(-1)
    shift = S_u.max(axis=-1, keepdims=True)
    Sh_l = S_l - shift; Sh_u = S_u - shift
    E_l = np.exp(Sh_l); E_u = np.exp(Sh_u)
    SumE_l = E_l.sum(-1, keepdims=True); SumE_u = E_u.sum(-1, keepdims=True)
    SumE_l = np.maximum(SumE_l, 1e-9)
    Inv_l = 1.0/SumE_u; Inv_u = 1.0/SumE_l
    Inv_b_l = np.broadcast_to(Inv_l, E_l.shape); Inv_b_u = np.broadcast_to(Inv_u, E_u.shape)
    cA1 = E_l*Inv_b_l; cA2 = E_l*Inv_b_u; cA3 = E_u*Inv_b_l; cA4 = E_u*Inv_b_u
    A_l = np.maximum(np.minimum(np.minimum(cA1,cA2),np.minimum(cA3,cA4)), 0.0)
    A_u = np.minimum(np.maximum(np.maximum(cA1,cA2),np.maximum(cA3,cA4)), 1.0)
    A_e_l = A_l[..., None]; A_e_u = A_u[..., None]
    V_e_l = V_l[:, None, :, :]; V_e_u = V_u[:, None, :, :]
    cO1 = A_e_l*V_e_l; cO2 = A_e_l*V_e_u; cO3 = A_e_u*V_e_l; cO4 = A_e_u*V_e_u
    OP_l = np.minimum(np.minimum(cO1,cO2),np.minimum(cO3,cO4))
    OP_u = np.maximum(np.maximum(cO1,cO2),np.maximum(cO3,cO4))
    O_l = OP_l.sum(2); O_u = OP_u.sum(2)
    O_cat_l = O_l.transpose(1,0,2).reshape(N, E)
    O_cat_u = O_u.transpose(1,0,2).reshape(N, E)
    out_l, out_u = ibp_lin(O_cat_l, O_cat_u, Wo, bo)
    return dict(Q=(Q_l,Q_u),K=(K_l,K_u),V=(V_l,V_u),S=(S_l,S_u),shift=shift,
                S_shifted=(Sh_l,Sh_u),E=(E_l,E_u),SumE=(SumE_l,SumE_u),
                Inv=(Inv_l,Inv_u),A=(A_l,A_u),O=(O_l,O_u),out=(out_l,out_u))

def ibp_mlp_block(x_l, x_u, mlp):
    W1 = mlp.fc1.weight.detach().cpu().numpy(); b1 = mlp.fc1.bias.detach().cpu().numpy()
    W2 = mlp.fc2.weight.detach().cpu().numpy(); b2 = mlp.fc2.bias.detach().cpu().numpy()
    h_l, h_u = ibp_lin(x_l, x_u, W1, b1)
    r_l, r_u = ibp_relu(h_l, h_u)
    o_l, o_u = ibp_lin(r_l, r_u, W2, b2)
    return o_l, o_u, (h_l, h_u)

def ibp_block(x_l, x_u, blk):
    g1 = blk.norm1.weight.detach().cpu().numpy(); eps = blk.norm1.eps
    n1_l, n1_u = ibp_rmsnorm(x_l, x_u, g1, eps)
    ibp_attn  = ibp_attention_bounds(n1_l, n1_u, blk.attn)
    a_l, a_u  = ibp_attn['out']
    r1_l, r1_u = x_l + a_l, x_u + a_u
    g2 = blk.norm2.weight.detach().cpu().numpy()
    n2_l, n2_u = ibp_rmsnorm(r1_l, r1_u, g2, eps)
    m_l, m_u, mlp_pre = ibp_mlp_block(n2_l, n2_u, blk.mlp)
    r2_l, r2_u = r1_l + m_l, r1_u + m_u
    return (r2_l, r2_u), dict(n1=(n1_l,n1_u), attn=ibp_attn, r1=(r1_l,r1_u),
                              n2=(n2_l,n2_u), mlp_pre=mlp_pre, m=(m_l,m_u))

def ibp_patch_embed(img_l, img_u, conv):
    Wp = conv.weight.clamp(min=0).detach().cpu().numpy()
    Wn = conv.weight.clamp(max=0).detach().cpu().numpy()
    b  = conv.bias.detach().cpu().numpy() if conv.bias is not None else None
    Wp_t = torch.from_numpy(Wp).float(); Wn_t = torch.from_numpy(Wn).float()
    img_l_t = torch.from_numpy(img_l).float(); img_u_t = torch.from_numpy(img_u).float()
    s, p = conv.stride, conv.padding
    l = F.conv2d(img_l_t, Wp_t, None, s, p) + F.conv2d(img_u_t, Wn_t, None, s, p)
    u = F.conv2d(img_u_t, Wp_t, None, s, p) + F.conv2d(img_l_t, Wn_t, None, s, p)
    if b is not None:
        b_t = torch.from_numpy(b).view(1, -1, 1, 1).float()
        l = l + b_t; u = u + b_t
    l = l.flatten(2).transpose(1, 2).numpy()
    u = u.flatten(2).transpose(1, 2).numpy()
    return l, u

def ibp_vit(model, img_lo, img_up):
    pe_l, pe_u = ibp_patch_embed(img_lo, img_up, model.patch_embed.proj)
    pos = model.pos_embed.detach().cpu().numpy()
    x_l = pe_l + pos; x_u = pe_u + pos
    stages = []
    for blk in model.blocks:
        (x_l_new, x_u_new), info = ibp_block(x_l[0], x_u[0], blk)
        stages.append(dict(input=(x_l[0], x_u[0]), info=info, output=(x_l_new, x_u_new)))
        x_l = x_l_new[None]; x_u = x_u_new[None]
    g = model.norm.weight.detach().cpu().numpy(); eps = model.norm.eps
    nf_l, nf_u = ibp_rmsnorm(x_l[0], x_u[0], g, eps)
    pool_l = nf_l.mean(0); pool_u = nf_u.mean(0)
    Wh = model.head.weight.detach().cpu().numpy(); bh = model.head.bias.detach().cpu().numpy()
    log_l, log_u = ibp_lin(pool_l[None], pool_u[None], Wh, bh)
    return log_l, log_u, dict(stages=stages, norm_final=(nf_l, nf_u),
                              pool=(pool_l, pool_u), logits=(log_l[0], log_u[0]))

In [8]:
# ── MILP encoders for ViT (notebooks 15-17) ─────────────────────────────
def encode_rmsnorm(milp, x_vars, x_lo, x_up, gamma, eps_rms, n_pieces=8, prefix=''):
    D = len(x_vars)
    x_lo = np.asarray(x_lo, dtype=float); x_up = np.asarray(x_up, dtype=float)
    gamma = np.asarray(gamma, dtype=float)
    xsq_vars = []; xsq_lo_arr = np.zeros(D); xsq_up_arr = np.zeros(D)
    for i in range(D):
        a, b = float(x_lo[i]), float(x_up[i])
        if abs(a-b) < 1e-12:
            v = milp.addVar(lb=a*a, ub=a*a, name=f'{prefix}xsq_{i}')
            milp.addConstr(v == a*a); xsq_lo_arr[i]=xsq_up_arr[i]=a*a
            xsq_vars.append(v); continue
        br = pwl_square(a, b, n=n_pieces)
        if a <= 0 <= b: xsq_lo_arr[i] = 0.0
        else: xsq_lo_arr[i] = min(a*a, b*b)
        xsq_up_arr[i] = max(a*a, b*b)
        v = milp.addVar(lb=xsq_lo_arr[i]-1e-3, ub=xsq_up_arr[i]+1e-3, name=f'{prefix}xsq_{i}')
        add_pwl_bracket(milp, x_vars[i], v, br, prefix=f'{prefix}xsq_{i}_')
        xsq_vars.append(v)
    mlo = float(xsq_lo_arr.mean()); mup = float(xsq_up_arr.mean())
    mean_xsq = milp.addVar(lb=mlo, ub=mup, name=f'{prefix}msq')
    milp.addConstr(mean_xsq == gp.quicksum(xsq_vars) / D)
    dlo, dup = mlo + eps_rms, mup + eps_rms
    denom = milp.addVar(lb=dlo, ub=dup, name=f'{prefix}den')
    milp.addConstr(denom == mean_xsq + eps_rms)
    if dlo <= 0: raise ValueError('eps_rms too small')
    br_inv = pwl_inv_sqrt_pos(dlo, dup, n=n_pieces)
    inv_lo, inv_up = 1.0/math.sqrt(dup), 1.0/math.sqrt(dlo)
    inv_rms = milp.addVar(lb=inv_lo-1e-3, ub=inv_up+1e-3, name=f'{prefix}inv')
    add_pwl_bracket(milp, denom, inv_rms, br_inv, prefix=f'{prefix}inv_')
    y_vars = []
    for i in range(D):
        a_l, a_u = float(x_lo[i]), float(x_up[i])
        if abs(a_l-a_u) < 1e-12:
            zlo = a_l*inv_lo if a_l>=0 else a_l*inv_up
            zup = a_l*inv_up if a_l>=0 else a_l*inv_lo
            z = milp.addVar(lb=zlo, ub=zup, name=f'{prefix}xi_{i}')
            milp.addConstr(z == a_l * inv_rms)
        else:
            z = add_mccormick(milp, x_vars[i], inv_rms, a_l, a_u, inv_lo, inv_up,
                              prefix=f'{prefix}xi_{i}_')
        g = float(gamma[i])
        ylo = g*z.LB if g>=0 else g*z.UB; yup = g*z.UB if g>=0 else g*z.LB
        y = milp.addVar(lb=ylo, ub=yup, name=f'{prefix}y_{i}')
        milp.addConstr(y == g * z); y_vars.append(y)
    return y_vars

def encode_mhsa(milp, x_vars, x_lo, x_up, attn, ibp_b, n_pieces=8, prefix=''):
    N = len(x_vars); E = len(x_vars[0])
    H, D = attn.num_heads, attn.head_dim; scale = attn.scale
    Wq = attn.W_q.weight.detach().cpu().numpy()
    Wk = attn.W_k.weight.detach().cpu().numpy()
    Wv = attn.W_v.weight.detach().cpu().numpy()
    bv = attn.W_v.bias.detach().cpu().numpy() if attn.W_v.bias is not None else np.zeros(E)
    Wo = attn.W_o.weight.detach().cpu().numpy()
    bo = attn.W_o.bias.detach().cpu().numpy() if attn.W_o.bias is not None else np.zeros(E)
    Q_l,Q_u = ibp_b['Q']; K_l,K_u = ibp_b['K']; V_l,V_u = ibp_b['V']
    S_l,S_u = ibp_b['S']; shift = ibp_b['shift']
    Sh_l,Sh_u = ibp_b['S_shifted']; E_l_b,E_u_b = ibp_b['E']
    SumE_l,SumE_u = ibp_b['SumE']; Inv_l_b,Inv_u_b = ibp_b['Inv']
    A_l_b,A_u_b = ibp_b['A']
    def linproj(W, b_arr, p):
        E_out = W.shape[0]; out = []
        for i in range(N):
            row = []
            for e in range(E_out):
                expr = gp.quicksum(W[e, ep]*x_vars[i][ep] for ep in range(E)) + float(b_arr[e])
                v = milp.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f'{p}_{i}_{e}')
                milp.addConstr(v == expr); row.append(v)
            out.append(row)
        return out
    Qv = linproj(Wq, np.zeros(E), f'{prefix}Q')
    Kv = linproj(Wk, np.zeros(E), f'{prefix}K')
    Vv = linproj(Wv, bv,           f'{prefix}V')
    def to_h(v): return [[[v[i][h*D + d] for d in range(D)] for i in range(N)] for h in range(H)]
    Qh, Kh, Vh = to_h(Qv), to_h(Kv), to_h(Vv)
    Sv = [[[None]*N for _ in range(N)] for _ in range(H)]
    for h in range(H):
        for i in range(N):
            for j in range(N):
                z_terms = []
                for d in range(D):
                    z = add_mccormick(milp, Qh[h][i][d], Kh[h][j][d],
                        float(Q_l[h,i,d]), float(Q_u[h,i,d]),
                        float(K_l[h,j,d]), float(K_u[h,j,d]),
                        prefix=f'{prefix}qk_{h}_{i}_{j}_{d}_')
                    z_terms.append(z)
                s = milp.addVar(lb=float(S_l[h,i,j])-1e-3, ub=float(S_u[h,i,j])+1e-3,
                                name=f'{prefix}S_{h}_{i}_{j}')
                milp.addConstr(s == scale*gp.quicksum(z_terms))
                Sv[h][i][j] = s
    Av = [[[None]*N for _ in range(N)] for _ in range(H)]
    for h in range(H):
        for i in range(N):
            sh_const = float(shift[h,i,0])
            E_row = []
            for j in range(N):
                shl, shu = float(Sh_l[h,i,j]), float(Sh_u[h,i,j])
                if shu - shl < 1e-9:
                    ec = math.exp(0.5*(shl+shu))
                    e = milp.addVar(lb=ec-1e-9, ub=ec+1e-9, name=f'{prefix}E_{h}_{i}_{j}')
                    milp.addConstr(e == ec)
                else:
                    br = pwl_exp(shl, shu, n=n_pieces)
                    ssh = milp.addVar(lb=shl, ub=shu, name=f'{prefix}Ssh_{h}_{i}_{j}')
                    milp.addConstr(ssh == Sv[h][i][j] - sh_const)
                    e = milp.addVar(lb=float(E_l_b[h,i,j])-1e-6, ub=float(E_u_b[h,i,j])+1e-6,
                                    name=f'{prefix}E_{h}_{i}_{j}')
                    add_pwl_bracket(milp, ssh, e, br, prefix=f'{prefix}E_{h}_{i}_{j}_')
                E_row.append(e)
            sel, seu = float(SumE_l[h,i,0]), float(SumE_u[h,i,0])
            sum_e = milp.addVar(lb=max(sel,1e-9), ub=seu, name=f'{prefix}SE_{h}_{i}')
            milp.addConstr(sum_e == gp.quicksum(E_row))
            il, iu = float(Inv_l_b[h,i,0]), float(Inv_u_b[h,i,0])
            if seu - sel < 1e-9:
                ic = 1.0 / (0.5*(sel+seu))
                inv_e = milp.addVar(lb=ic-1e-9, ub=ic+1e-9, name=f'{prefix}I_{h}_{i}')
                milp.addConstr(inv_e == ic)
            else:
                br_i = pwl_inv_pos(max(sel,1e-9), seu, n=n_pieces)
                inv_e = milp.addVar(lb=il-1e-6, ub=iu+1e-6, name=f'{prefix}I_{h}_{i}')
                add_pwl_bracket(milp, sum_e, inv_e, br_i, prefix=f'{prefix}I_{h}_{i}_')
            for j in range(N):
                a = add_mccormick(milp, E_row[j], inv_e,
                    float(E_l_b[h,i,j]), float(E_u_b[h,i,j]), il, iu,
                    prefix=f'{prefix}A_{h}_{i}_{j}_', y_lb=0.0, y_ub=1.0)
                Av[h][i][j] = a
    Ov = [[[None]*D for _ in range(N)] for _ in range(H)]
    for h in range(H):
        for i in range(N):
            for d in range(D):
                z_terms = []
                for j in range(N):
                    z = add_mccormick(milp, Av[h][i][j], Vh[h][j][d],
                        float(A_l_b[h,i,j]), float(A_u_b[h,i,j]),
                        float(V_l[h,j,d]), float(V_u[h,j,d]),
                        prefix=f'{prefix}av_{h}_{i}_{j}_{d}_')
                    z_terms.append(z)
                o = milp.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f'{prefix}O_{h}_{i}_{d}')
                milp.addConstr(o == gp.quicksum(z_terms))
                Ov[h][i][d] = o
    O_cat = [[Ov[h][i][d] for h in range(H) for d in range(D)] for i in range(N)]
    out_vars = []
    for i in range(N):
        row = []
        for e in range(E):
            expr = gp.quicksum(Wo[e, ep]*O_cat[i][ep] for ep in range(E)) + float(bo[e])
            v = milp.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f'{prefix}out_{i}_{e}')
            milp.addConstr(v == expr); row.append(v)
        out_vars.append(row)
    return out_vars

def encode_mlp_block(milp, x_vars, x_lo, x_up, mlp, ibp_pre, prefix=''):
    N = len(x_vars); E = len(x_vars[0])
    W1 = mlp.fc1.weight.detach().cpu().numpy(); b1 = mlp.fc1.bias.detach().cpu().numpy()
    W2 = mlp.fc2.weight.detach().cpu().numpy(); b2 = mlp.fc2.bias.detach().cpu().numpy()
    Eh = W1.shape[0]
    h_l_arr, h_u_arr = ibp_pre
    out_vars = []
    for i in range(N):
        h_vars = []
        for e in range(Eh):
            expr = gp.quicksum(W1[e, ep]*x_vars[i][ep] for ep in range(E)) + float(b1[e])
            v = milp.addVar(lb=float(h_l_arr[i,e])-1e-3, ub=float(h_u_arr[i,e])+1e-3,
                            name=f'{prefix}h_{i}_{e}')
            milp.addConstr(v == expr); h_vars.append(v)
        r_vars = [add_relu_bigM(milp, h_vars[e], float(h_l_arr[i,e]), float(h_u_arr[i,e]),
                                prefix=f'{prefix}r_{i}_{e}_') for e in range(Eh)]
        row = []
        for e in range(E):
            expr = gp.quicksum(W2[e, ep]*r_vars[ep] for ep in range(Eh)) + float(b2[e])
            v = milp.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f'{prefix}o_{i}_{e}')
            milp.addConstr(v == expr); row.append(v)
        out_vars.append(row)
    return out_vars

def encode_block(milp, x_vars, x_lo, x_up, blk, ibp_b, n_pieces=8, prefix=''):
    N = len(x_vars); E = len(x_vars[0])
    g1 = blk.norm1.weight.detach().cpu().numpy(); eps = blk.norm1.eps
    n1_l, n1_u = ibp_b['n1']
    n1_vars = [encode_rmsnorm(milp, x_vars[i], x_lo[i], x_up[i], g1, eps,
                              n_pieces=n_pieces, prefix=f'{prefix}n1_{i}_')
               for i in range(N)]
    a_vars = encode_mhsa(milp, n1_vars, n1_l, n1_u, blk.attn, ibp_b['attn'],
                         n_pieces=n_pieces, prefix=f'{prefix}atn_')
    r1_l, r1_u = ibp_b['r1']
    r1_vars = []
    for i in range(N):
        row = []
        for e in range(E):
            v = milp.addVar(lb=float(r1_l[i,e]), ub=float(r1_u[i,e]),
                            name=f'{prefix}r1_{i}_{e}')
            milp.addConstr(v == x_vars[i][e] + a_vars[i][e]); row.append(v)
        r1_vars.append(row)
    g2 = blk.norm2.weight.detach().cpu().numpy()
    n2_l, n2_u = ibp_b['n2']
    n2_vars = [encode_rmsnorm(milp, r1_vars[i], r1_l[i], r1_u[i], g2, eps,
                              n_pieces=n_pieces, prefix=f'{prefix}n2_{i}_')
               for i in range(N)]
    m_vars = encode_mlp_block(milp, n2_vars, n2_l, n2_u, blk.mlp,
                               ibp_b['mlp_pre'], prefix=f'{prefix}mlp_')
    r2_vars = []
    for i in range(N):
        row = []
        for e in range(E):
            v = milp.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f'{prefix}r2_{i}_{e}')
            milp.addConstr(v == r1_vars[i][e] + m_vars[i][e]); row.append(v)
        r2_vars.append(row)
    return r2_vars

def encode_vit(milp, img_vars, img_lo, img_up, model, n_pieces=8, prefix=''):
    log_l, log_u, ibp_full = ibp_vit(model, img_lo, img_up)
    conv = model.patch_embed.proj
    Wc = conv.weight.detach().cpu().numpy()
    bc = conv.bias.detach().cpu().numpy() if conv.bias is not None else np.zeros(Wc.shape[0])
    C, Himg, Wimg = img_vars.shape
    P = conv.kernel_size[0]; H_out = Himg // P; W_out = Wimg // P; N = H_out * W_out
    E = Wc.shape[0]
    pe_vars = []
    for i_h in range(H_out):
        for i_w in range(W_out):
            row = []
            for e in range(E):
                terms = []
                for c_ in range(C):
                    for p_h in range(P):
                        for p_w in range(P):
                            w_ = float(Wc[e, c_, p_h, p_w])
                            if w_ != 0.0:
                                terms.append(w_ * img_vars[c_, i_h*P + p_h, i_w*P + p_w])
                expr = gp.quicksum(terms) + float(bc[e])
                v = milp.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY,
                                name=f'{prefix}pe_{i_h}_{i_w}_{e}')
                milp.addConstr(v == expr); row.append(v)
            pe_vars.append(row)
    pos = model.pos_embed.detach().cpu().numpy()[0]
    x_vars = []
    for i in range(N):
        row = []
        for e in range(E):
            v = milp.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f'{prefix}xi_{i}_{e}')
            milp.addConstr(v == pe_vars[i][e] + float(pos[i, e]))
            row.append(v)
        x_vars.append(row)
    pe_l, pe_u = ibp_patch_embed(img_lo, img_up, conv)
    x_l = pe_l[0] + pos; x_u = pe_u[0] + pos
    for bi, blk in enumerate(model.blocks):
        ibp_b = ibp_full['stages'][bi]['info']
        x_vars = encode_block(milp, x_vars, x_l, x_u, blk, ibp_b,
                               n_pieces=n_pieces, prefix=f'{prefix}b{bi}_')
        x_l, x_u = ibp_full['stages'][bi]['output']
    g = model.norm.weight.detach().cpu().numpy(); eps = model.norm.eps
    nf_l, nf_u = ibp_full['norm_final']
    nf_vars = [encode_rmsnorm(milp, x_vars[i], x_l[i], x_u[i], g, eps,
                              n_pieces=n_pieces, prefix=f'{prefix}nf_{i}_')
               for i in range(N)]
    pool_l, pool_u = ibp_full['pool']
    pool_vars = []
    for e in range(E):
        v = milp.addVar(lb=float(pool_l[e])-1e-3, ub=float(pool_u[e])+1e-3,
                        name=f'{prefix}pool_{e}')
        milp.addConstr(v == gp.quicksum(nf_vars[i][e] for i in range(N)) / N)
        pool_vars.append(v)
    Wh = model.head.weight.detach().cpu().numpy(); bh = model.head.bias.detach().cpu().numpy()
    K = Wh.shape[0]
    log_vars = []
    for k in range(K):
        v = milp.addVar(lb=float(log_l[0,k])-1e-3, ub=float(log_u[0,k])+1e-3,
                        name=f'{prefix}log_{k}')
        milp.addConstr(v == gp.quicksum(float(Wh[k,e])*pool_vars[e] for e in range(E)) + float(bh[k]))
        log_vars.append(v)
    return log_vars, ibp_full

def verify_vit_milp(model, x0, eps, true_label, n_pieces=8, time_limit=300.0,
                    target_class=None):
    if not HAS_GUROBI: return dict(status='skipped', reason='no gurobi')
    img_lo = np.clip(x0 - eps, 0.0, 1.0)
    img_up = np.clip(x0 + eps, 0.0, 1.0)
    targets = [target_class] if target_class is not None \
              else [c for c in range(10) if c != true_label]
    m = gp.Model('vit_verify')
    m.setParam('OutputFlag', 0); m.setParam('TimeLimit', time_limit)
    C, Himg, Wimg = x0.shape
    img_vars = np.empty((C, Himg, Wimg), dtype=object)
    for c in range(C):
        for h in range(Himg):
            for w in range(Wimg):
                img_vars[c, h, w] = m.addVar(lb=float(img_lo[c, h, w]),
                                             ub=float(img_up[c, h, w]),
                                             name=f'img_{c}_{h}_{w}')
    log_vars, _ = encode_vit(m, img_vars, img_lo[None], img_up[None], model,
                              n_pieces=n_pieces)
    m.update()
    times = {}; counterexample = None; worst = float('inf'); worst_c = None
    overall_t0 = time.time()
    for c in targets:
        if time.time() - overall_t0 > time_limit:
            return dict(status='inconclusive', reason='time_limit',
                        worst_margin=worst, worst_class=worst_c,
                        time_per_class=times, n_pieces=n_pieces)
        margin_var = m.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f'mar_{c}')
        m.addConstr(margin_var == log_vars[true_label] - log_vars[c])
        m.setObjective(margin_var, GRB.MINIMIZE)
        t0 = time.time()
        per_target_limit = max(5.0, time_limit - (time.time() - overall_t0))
        m.setParam('TimeLimit', per_target_limit)
        m.optimize(); times[c] = time.time() - t0
        st = m.Status
        if st == GRB.INFEASIBLE: return dict(status='error', reason='infeasible')
        if st in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
            margin = margin_var.X
        elif st == GRB.TIME_LIMIT and m.SolCount > 0:
            margin = margin_var.X
        else:
            return dict(status='inconclusive', reason=f'status={st}',
                        time_per_class=times, n_pieces=n_pieces)
        if margin < worst:
            worst = margin; worst_c = c
            if margin < 0:
                ce = np.array([[[img_vars[ci,hi,wi].X for wi in range(Wimg)]
                                for hi in range(Himg)] for ci in range(C)])
                counterexample = ce
        m.remove(margin_var); m.update()
        if margin < 0:
            return dict(status='falsified', worst_margin=margin, worst_class=worst_c,
                        counterexample=counterexample, time_per_class=times,
                        n_pieces=n_pieces)
    return dict(status='verified', worst_margin=worst, worst_class=worst_c,
                time_per_class=times, n_pieces=n_pieces)

In [9]:
# ── PGD L∞ attack (notebook 13) ─────────────────────────────────────────
def pgd_linf(model, x, y, eps, n_steps=100, n_restarts=5, alpha=None, clamp=(0.0, 1.0)):
    if alpha is None: alpha = 0.1 * eps
    model.eval()
    lo, hi = clamp
    best_loss = torch.full((len(x),), -1e9, device=x.device)
    best_adv  = x.clone()
    for r in range(n_restarts):
        delta = torch.zeros_like(x) if r == 0 else torch.empty_like(x).uniform_(-eps, eps)
        delta.requires_grad_(True)
        for _ in range(n_steps):
            adv  = (x + delta).clamp(lo, hi)
            loss = F.cross_entropy(model(adv), y, reduction='sum')
            grad, = torch.autograd.grad(loss, delta)
            with torch.no_grad():
                delta = (delta + alpha * grad.sign()).clamp(-eps, eps)
                delta = ((x + delta).clamp(lo, hi) - x).detach()
            delta.requires_grad_(True)
        with torch.no_grad():
            adv = (x + delta).clamp(lo, hi)
            per = F.cross_entropy(model(adv), y, reduction='none')
            mask = per > best_loss
            best_loss[mask] = per[mask]
            best_adv[mask]  = adv[mask]
    with torch.no_grad():
        success = model(best_adv).argmax(1) != y
    return best_adv, success

In [19]:
# ── Drive paths and Lipschitz constants ─────────────────────────────────
DRIVE_BASE = Path('/content/drive/My Drive/thesis-formal-verification')                                                              
CKPTS = {                                                                                                                              
    'standard':  DRIVE_BASE / 'runs/vit_tiny_standard/model.pt',
    'lipmargin': DRIVE_BASE / 'runs/vit_tiny_lipmargin/model.pt',                                                                      
}                                                         
RESULTS_DRIVE = DRIVE_BASE / 'results/vit_p2'

# Per-model Lipschitz files written by notebook 11 — one JSON per model.
LIP_PATHS = {
    'standard':  DRIVE_BASE / 'runs/vit_tiny_standard/lipschitz.json',
    'lipmargin': DRIVE_BASE / 'runs/vit_tiny_lipmargin/lipschitz.json',
}

def load_lipschitz_consts():
    """Return {model_name: L_total} loaded from per-model JSON files.
    Falls back to None if no files found (caller computes crude bound)."""
    consts = {}
    for name, path in LIP_PATHS.items():
        if not path.exists():
            print(f'  [{name}] no file at {path}')
            continue
        try:
            data = json.loads(path.read_text())
        except Exception as e:
            print(f'  [{name}] failed to parse {path}: {e}')
            continue
        # Accept three layouts:
        #   1) {"L_total": ...}                  (flat, per-model file — preferred)
        #   2) {name: {"L_total": ...}}          (legacy nested)
        #   3) {"global": {"L_total": ...}}      (component-breakdown layout)
        L = None
        if isinstance(data, dict):
            if 'L_total' in data:
                L = float(data['L_total'])
            elif name in data and isinstance(data[name], dict) and 'L_total' in data[name]:
                L = float(data[name]['L_total'])
            elif 'global' in data and isinstance(data['global'], dict) and 'L_total' in data['global']:
                L = float(data['global']['L_total'])
        if L is None:
            print(f'  [{name}] no L_total key in {path}; top-level keys={list(data.keys())[:8]}')
            continue
        consts[name] = L
        print(f'  [{name}] L_total = {L:.3e}  (from {path.name})')
    if consts:
        return consts
    print('No Lipschitz files loaded; will compute crude bound on the fly.')
    return None

def crude_lipschitz_bound(model: ViTTiny) -> float:
    """
    Conservative global L from Plan 2 §2.3.  Not as tight as notebook 11's
    Kim-et-al attention bound but always sound.  Used only as a fallback.
    """
    def sigma(W): return float(torch.linalg.svdvals(W)[0])
    # Patch embed: reshape conv to 2D matrix
    Wpe = model.patch_embed.proj.weight
    Wpe_2d = Wpe.reshape(Wpe.shape[0], -1)
    L_pe = sigma(Wpe_2d)
    L_blocks = 1.0
    for blk in model.blocks:
        # crude per-block bound: (1 + L_norm * L_attn) * (1 + L_norm * L_mlp)
        D = model.cfg['embed_dim']
        L_norm = float(blk.norm1.weight.abs().max().item()) * math.sqrt(D / blk.norm1.eps)
        # L_mlp ≤ σ(W2)·σ(W1)
        L_mlp  = sigma(blk.mlp.fc2.weight) * sigma(blk.mlp.fc1.weight)
        # L_attn ≤ σ(W_O) · sqrt(H · L_head²) (Kim et al., simplified)
        H = model.cfg['num_heads']; head_dim = D // H
        N = model.patch_embed.n_patches
        L_head = N**1.5 * sigma(blk.attn.W_q.weight) * sigma(blk.attn.W_k.weight) \
                       * sigma(blk.attn.W_v.weight) / math.sqrt(head_dim) \
                + N**0.5 * sigma(blk.attn.W_v.weight)
        L_mhsa = sigma(blk.attn.W_o.weight) * math.sqrt(H * L_head**2)
        L_blocks *= (1 + L_norm * L_mhsa) * (1 + L_norm * L_mlp)
    L_head = sigma(model.head.weight) / math.sqrt(model.patch_embed.n_patches)
    L_norm_final = float(model.norm.weight.abs().max().item()) * math.sqrt(model.cfg['embed_dim'] / model.norm.eps)
    return L_pe * L_blocks * L_norm_final * L_head

In [20]:
# ── Eval split (50 deterministic MNIST samples, seed=1234) ───────────────
def get_eval_set(n=50, seed=1234):
    tf = transforms.ToTensor()
    test_ds = torchvision.datasets.MNIST('/tmp/mnist', train=False, download=True, transform=tf)
    rng = np.random.default_rng(seed)
    idx = sorted(rng.choice(len(test_ds), size=n, replace=False).tolist())
    samples = []
    for i in idx:
        x, y = test_ds[i]
        samples.append((i, x.numpy(), int(y)))
    return samples

In [21]:
# ── Hybrid verifier ─────────────────────────────────────────────────────
def clean_margin(model: nn.Module, x0_np: np.ndarray, y: int) -> float:
    """Returns logit[y] - max_{c≠y} logit[c] for x0 (a single image, no batch)."""
    with torch.no_grad():
        z = model(torch.from_numpy(x0_np[None]).float().to(next(model.parameters()).device))[0]
        z = z.cpu().numpy()
        other = z.copy(); other[y] = -np.inf
        return float(z[y] - other.max())

def hybrid_verify(model: nn.Module, x0_np: np.ndarray, eps: float, y_true: int,
                  L_global: float, milp_timeout: float = 120.0,
                  n_pieces: int = 8) -> Dict[str, Any]:
    """
    Returns:
        method   : 'lipschitz' | 'ibp' | 'pgd' | 'milp' | 'milp_timeout'
        status   : 'verified' | 'falsified' | 'inconclusive' | 'misclassified'
        margins  : dict per stage (lipschitz_pre_margin, ibp_lb_y, ibp_ub_other,
                                   milp_worst_margin)
        times    : dict per stage in seconds
        ce       : counterexample image (numpy) if falsified, else None
    """
    out = dict(method=None, status=None, margins={}, times={}, ce=None)

    # 0. clean prediction
    cm = clean_margin(model, x0_np, y_true)
    out['margins']['clean'] = cm
    if cm <= 0:
        out['method'] = 'clean'; out['status'] = 'misclassified'
        return out

    # 1. Lipschitz pre-filter
    t0 = time.time()
    D = int(np.prod(x0_np.shape))
    threshold = L_global * eps * math.sqrt(D)
    out['margins']['lipschitz_threshold'] = threshold
    out['times']['lipschitz'] = time.time() - t0
    if cm > threshold:
        out['method'] = 'lipschitz'; out['status'] = 'verified'
        return out

    # 2. IBP propagation
    t0 = time.time()
    img_lo = np.clip(x0_np[None] - eps, 0, 1).astype(np.float64)
    img_up = np.clip(x0_np[None] + eps, 0, 1).astype(np.float64)
    log_l, log_u, _ = ibp_vit(model, img_lo, img_up)
    lb_y = float(log_l[0, y_true])
    other = log_u[0].copy(); other[y_true] = -np.inf
    ub_other = float(other.max())
    out['margins']['ibp_lb_y'] = lb_y; out['margins']['ibp_ub_other'] = ub_other
    out['times']['ibp'] = time.time() - t0
    if lb_y > ub_other:
        out['method'] = 'ibp'; out['status'] = 'verified'
        return out

    # 3. PGD attack
    t0 = time.time()
    dev = next(model.parameters()).device
    x_t = torch.from_numpy(x0_np[None]).float().to(dev)
    y_t = torch.tensor([y_true], device=dev)
    adv, success = pgd_linf(model, x_t, y_t, eps, n_steps=100, n_restarts=5)
    out['times']['pgd'] = time.time() - t0
    if bool(success.item()):
        out['method'] = 'pgd'; out['status'] = 'falsified'
        out['ce'] = adv[0].detach().cpu().numpy()
        return out

    # 4. MILP (last resort)
    if not HAS_GUROBI:
        out['method'] = 'milp_skipped'; out['status'] = 'inconclusive'
        return out
    t0 = time.time()
    try:
        verdict = verify_vit_milp(model, x0_np, eps, y_true,
                                   n_pieces=n_pieces, time_limit=milp_timeout)
    except Exception as e:
        out['method'] = 'milp_error'; out['status'] = 'inconclusive'
        out['margins']['milp_error'] = str(e)
        out['times']['milp'] = time.time() - t0
        return out
    out['times']['milp'] = time.time() - t0
    out['margins']['milp_worst_margin'] = verdict.get('worst_margin')
    if verdict.get('status') == 'verified':
        out['method'] = 'milp'; out['status'] = 'verified'
    elif verdict.get('status') == 'falsified':
        out['method'] = 'milp'; out['status'] = 'falsified'
        out['ce'] = verdict.get('counterexample')
    else:
        out['method'] = 'milp_timeout'; out['status'] = 'inconclusive'
    return out

In [22]:
# ── Run hybrid verifier on the eval set ─────────────────────────────────
EPS_LIST = [0.01, 0.03, 0.1]
N_SAMPLES = 50
MILP_TIMEOUT = 120.0
N_PIECES = 6   # smaller for speed

samples = get_eval_set(n=N_SAMPLES, seed=1234)
print(f'Loaded {len(samples)} eval samples')

lip_consts = load_lipschitz_consts() or {}

rows = []
for model_name, ckpt in CKPTS.items():
    if not ckpt.exists():
        print(f'[skip] {model_name}: {ckpt} not found'); continue
    print(f'\n══ {model_name} ══════════════════════════════════════')
    model = load_vit(ckpt, device=str(device))
    L = lip_consts.get(model_name)
    if L is None:
        L = crude_lipschitz_bound(model)
        print(f'  Lipschitz (crude) = {L:.3e}')
    else:
        print(f'  Lipschitz (saved) = {L:.3e}')
    for eps in EPS_LIST:
        print(f'  ε = {eps}')
        method_counts = {}
        for (idx, x, y) in samples:
            r = hybrid_verify(model, x, eps, y, L_global=L,
                              milp_timeout=MILP_TIMEOUT, n_pieces=N_PIECES)
            row = dict(model=model_name, eps=eps, idx=idx, true_label=y,
                       method=r['method'], status=r['status'],
                       t_lip = r['times'].get('lipschitz', 0.0),
                       t_ibp = r['times'].get('ibp', 0.0),
                       t_pgd = r['times'].get('pgd', 0.0),
                       t_milp = r['times'].get('milp', 0.0),
                       clean_margin   = r['margins'].get('clean'),
                       lip_threshold  = r['margins'].get('lipschitz_threshold'),
                       milp_margin    = r['margins'].get('milp_worst_margin'))
            rows.append(row)
            method_counts[r['method']] = method_counts.get(r['method'], 0) + 1
        print(f'    methods: {method_counts}')

df = pd.DataFrame(rows)
print(f'\nTotal rows: {len(df)}')
print(df.groupby(['model', 'eps', 'status']).size().unstack(fill_value=0))

Loaded 50 eval samples
  [standard] L_total = 3.177e+21  (from lipschitz.json)
  [lipmargin] L_total = 3.674e+23  (from lipschitz.json)

══ standard ══════════════════════════════════════
  Lipschitz (saved) = 3.177e+21
  ε = 0.01
    methods: {'milp_error': 46, 'clean': 2, 'pgd': 2}
  ε = 0.03
    methods: {'pgd': 16, 'milp_error': 32, 'clean': 2}
  ε = 0.1
    methods: {'pgd': 48, 'clean': 2}

══ lipmargin ══════════════════════════════════════
  Lipschitz (saved) = 3.674e+23
  ε = 0.01
    methods: {'clean': 46, 'pgd': 4}
  ε = 0.03
    methods: {'clean': 46, 'pgd': 4}
  ε = 0.1
    methods: {'clean': 46, 'pgd': 4}

Total rows: 300
status          falsified  inconclusive  misclassified
model     eps                                         
lipmargin 0.01          4             0             46
          0.03          4             0             46
          0.10          4             0             46
standard  0.01          2            46              2
          0.03         16  

In [18]:
# ── Save per-sample CSV + aggregate report ──────────────────────────────
out_dir = Path('results/vit_p2'); out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / 'hybrid_results.csv'
df.to_csv(csv_path, index=False)
print(f'Saved → {csv_path}')

# Aggregate counts of (model, eps, method)
agg = df.groupby(['model', 'eps', 'method']).size().reset_index(name='n')
agg_path = out_dir / 'hybrid_method_breakdown.csv'
agg.to_csv(agg_path, index=False)
print(f'Saved → {agg_path}')

# Summary JSON for the report
summary = {}
for (mn, eps), grp in df.groupby(['model', 'eps']):
    key = f'{mn}|eps={eps}'
    summary[key] = dict(
        n           = int(len(grp)),
        verified    = int((grp.status == 'verified').sum()),
        falsified   = int((grp.status == 'falsified').sum()),
        inconclusive= int((grp.status == 'inconclusive').sum()),
        misclassified= int((grp.status == 'misclassified').sum()),
        method_counts = grp['method'].value_counts().to_dict(),
        mean_total_time_s = float((grp.t_lip + grp.t_ibp + grp.t_pgd + grp.t_milp).mean()),
    )
sum_path = out_dir / 'hybrid_summary.json'
sum_path.write_text(json.dumps(summary, indent=2, default=float))
print(f'Saved → {sum_path}')

try:
    drive_out = Path('/content/drive/My Drive/thesis-formal-verification/results/vit_p2')
    drive_out.mkdir(parents=True, exist_ok=True)
    for p in (csv_path, agg_path, sum_path):
        (drive_out / p.name).write_text(p.read_text())
    print(f'Mirrored to drive: {drive_out}')
except Exception as e:
    print(f'(skipping drive mirror: {e})')

print('\nPhase 6 complete: hybrid verifier results saved.')
print('Next: Phase 7 (α,β-CROWN comparison; notebook 19).')

Saved → results/vit_p2/hybrid_results.csv
Saved → results/vit_p2/hybrid_method_breakdown.csv
Saved → results/vit_p2/hybrid_summary.json
Mirrored to drive: /content/drive/My Drive/thesis-formal-verification/results/vit_p2

Phase 6 complete: hybrid verifier results saved.
Next: Phase 7 (α,β-CROWN comparison; notebook 19).
